In [92]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score
# To save models
import math
import json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint
import joblib
from contextlib import contextmanager
from scipy.stats import randint, uniform
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import optuna
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
from sklearn.model_selection import ParameterSampler
from scipy.stats import randint, uniform
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import numpy as np



In [93]:
seed = 18

#df = pd.read_csv("../data/processed/df_v2")
#df = pd.read_csv("../data/processed/df_withinetacolum")
df = pd.read_csv("../data/processed/df")


df = df.sort_values("num_semana").reset_index(drop=True)
weeks = df["num_semana"].unique()

cut_w = int(len(weeks) * 0.8)
train_weeks = weeks[:cut_w]
test_weeks  = weeks[cut_w:]

train = df[df["num_semana"].isin(train_weeks)]
test  = df[df["num_semana"].isin(test_weeks)]

X_train, y_train = train.drop(columns=["y"]), train["y"]
X_test,  y_test  = test.drop(columns=["y"]),  test["y"]


In [94]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0,
    allow_writing_files=False
)

cb.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

In [95]:
model = cb
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_train:.2f} % de variacion Train")
print(f"R² (Coef. determinación): {r2_test:.2f} % de variacion Test")

MSE (Error cuadrático medio): 22.17
RMSE (Raíz del ECM): 4.71 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.78 % de variacion Train
R² (Coef. determinación): 0.72 % de variacion Test


In [96]:
# suponiendo que ya tienes train df (solo semanas train) como antes
weeks_train = train["num_semana"].unique()
cut_val = int(len(weeks_train) * 0.9)

tr_weeks  = weeks_train[:cut_val]
val_weeks = weeks_train[cut_val:]

train_tr  = train[train["num_semana"].isin(tr_weeks)]
train_val = train[train["num_semana"].isin(val_weeks)]

X_tr, y_tr   = train_tr.drop(columns=["y"]), train_tr["y"]
X_val, y_val = train_val.drop(columns=["y"]), train_val["y"]


In [97]:
rng = np.random.RandomState(seed)

param_dist = {
    "iterations": randint(800, 5000),
    "learning_rate": uniform(0.01, 0.09),
    "depth": randint(4, 11),
    "l2_leaf_reg": uniform(1, 9),
    "random_strength": uniform(1, 4),
    "bagging_temperature": uniform(0, 1),
    "border_count": [128],
}

best_score = -1e9
best_params = None
best_model = None

for params in ParameterSampler(param_dist, n_iter=50, random_state=rng):
    model = CatBoostRegressor(
        loss_function="RMSE",
        random_seed=seed,
        verbose=0,
        allow_writing_files=False,
        **params
    )

    model.fit(
        X_tr, y_tr,
        cat_features=["product"],
        eval_set=(X_val, y_val),
        early_stopping_rounds=200,
        use_best_model=True
    )

    preds = model.predict(X_val)
    score = r2_score(y_val, preds)

    if score > best_score:
        best_score = score
        best_params = params
        best_model = model

print("Mejor R2 (val):", best_score)
print("Mejores params:", best_params)


Mejor R2 (val): 0.7698986347136353
Mejores params: {'bagging_temperature': np.float64(0.8785092691266934), 'border_count': 128, 'depth': 7, 'iterations': 3870, 'l2_leaf_reg': np.float64(5.651050028148526), 'learning_rate': np.float64(0.09542883171754328), 'random_strength': np.float64(1.8960317736025645)}


In [98]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_train:.2f} % de variacion Train")
print(f"R² (Coef. determinación): {r2_test:.2f} % de variacion Test")

MSE (Error cuadrático medio): 22.13
RMSE (Raíz del ECM): 4.70 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.77 % de variacion Train
R² (Coef. determinación): 0.72 % de variacion Test


In [99]:
dump(model, open("../models/72_Cat_Boost_Regressor.pkl", "wb"))